# 02 · Población por hexágono (Censo 2020)

**Objetivo:** llevar el Censo de Población y Vivienda 2020 a hexágonos H3 de resolución 8 y clasificar cada hexágono según dónde vive la gente y dónde están los restaurantes.

- **Censo 2020 por manzana:** población, viviendas, empleo, autos y escolaridad.
- **Unidad común:** cada manzana se asigna al hexágono que contiene su centroide.
- **Salida:** `hex_res8_base_2020.parquet`, la tabla de población que usan los notebooks 04 y 05.

La población es de 2020 porque es el último censo con datos por manzana. La Encuesta Intercensal 2025 solo llega a nivel municipal.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import tempfile, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import h3
import mapas

ROOT = Path.cwd().parent
RAW, PROC, FIG = ROOT / "data/raw", ROOT / "data/processed", ROOT / "figures"
TMP = Path(tempfile.gettempdir()) / "inegi_tmp"
TMP.mkdir(exist_ok=True)

URL_MG = ("https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/"
          "productos/geografia/marcogeo/889463807469/22_queretaro.zip")
ZIP_CENSO = RAW / "ageb_mza_urbana_22_cpv2020_csv.zip"
MUNICIPIOS_ZMQ = {6: "Corregidora", 8: "Huimilpan", 11: "El Marqués", 14: "Querétaro"}
RES, BASE = 8, "2020-04"

## 1. Centroides de manzana

Si el notebook 00 los copió del proyecto anterior, se leen. Si no, se descarga el Marco Geoestadístico 2020 (62.7 MB, a una carpeta temporal fuera de Dropbox) y se calcula el centroide de cada manzana.

In [ ]:
SALIDA_MZA = PROC / "manzanas_zmq_2020.parquet"

if SALIDA_MZA.exists():
    mza = pd.read_parquet(SALIDA_MZA)
    print("Centroides leídos de", SALIDA_MZA.name)
else:
    import geopandas as gpd
    zip_mg = TMP / "22_queretaro.zip"
    if not zip_mg.exists():
        req = urllib.request.Request(URL_MG, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=600) as r, open(zip_mg, "wb") as f:
            while bloque := r.read(1 << 20):
                f.write(bloque)
    destino = TMP / "mg_22"
    with zipfile.ZipFile(zip_mg) as z:
        z.extractall(destino, members=[n for n in z.namelist() if Path(n).stem.lower() == "22m"])
    gdf = gpd.read_file(next(destino.rglob("22m.shp")))
    gdf.columns = [c.upper() if c != "geometry" else c for c in gdf.columns]
    gdf = gdf[pd.to_numeric(gdf["CVE_MUN"]).isin(MUNICIPIOS_ZMQ)]
    cent = gpd.GeoSeries(gdf.geometry.centroid, crs=gdf.crs).to_crs(4326)
    mza = pd.DataFrame({"CVEGEO": gdf["CVEGEO"].values, "lat": cent.y.values,
                        "lon": cent.x.values, "area_m2": gdf.geometry.area.values})
    mza.to_parquet(SALIDA_MZA, index=False)
print(f"Manzanas de la ZM: {len(mza):,}")

## 2. Censo 2020 por manzana

Se quitan los renglones de totales (`MZA = 0`) y se arma `CVEGEO`. Las variables son conteos, así que se pueden sumar por hexágono; la escolaridad se guarda como promedio ponderado por la población de 15 años y más.

| Variable | Qué es |
|---|---|
| `POBTOT` | Población total |
| `TVIVHAB` | Viviendas habitadas |
| `POCUPADA` | Población ocupada |
| `VPH_AUTOM` | Viviendas con automóvil |
| `P_15YMAS`, `GRAPROES` | Población de 15 años y más, y su grado promedio de escolaridad |

In [ ]:
with zipfile.ZipFile(ZIP_CENSO) as z:
    nombre = max((i for i in z.infolist() if i.filename.lower().endswith(".csv")), key=lambda i: i.file_size)
    censo = pd.read_csv(z.open(nombre), dtype=str, encoding="utf-8", low_memory=False)
censo.columns = censo.columns.str.upper()
censo = censo[pd.to_numeric(censo["MUN"]).isin(MUNICIPIOS_ZMQ) & (pd.to_numeric(censo["MZA"]) != 0)].copy()
censo["CVEGEO"] = (censo["ENTIDAD"].str.zfill(2) + censo["MUN"].str.zfill(3) + censo["LOC"].str.zfill(4)
                   + censo["AGEB"].str.zfill(4) + censo["MZA"].str.zfill(3))
VARS = ["POBTOT", "TVIVHAB", "POCUPADA", "VPH_AUTOM", "P_15YMAS", "GRAPROES"]
for v in VARS:
    censo[v] = pd.to_numeric(censo[v], errors="coerce")       # "*" (confidencial) queda como faltante
censo["ESC_X_POB"] = censo["GRAPROES"] * censo["P_15YMAS"]
print(f"Manzanas en el Censo: {len(censo):,} · población: {censo['POBTOT'].sum():,.0f}")
print("Valores ocultos o faltantes:", censo[VARS].isna().sum().to_dict())

## 3. Llevar todo a hexágonos

In [ ]:
def a_hex(df, lat="latitud", lon="longitud"):
    return df.assign(h3=[h3.latlng_to_cell(la, lo, RES) for la, lo in zip(df[lat], df[lon])])

mzc = a_hex(censo.merge(mza, on="CVEGEO", how="inner"), "lat", "lon")
print(f"Manzanas del Censo con centroide: {len(mzc) / len(censo):.1%}")
pob = mzc.groupby("h3")[["POBTOT", "TVIVHAB", "POCUPADA", "VPH_AUTOM", "P_15YMAS", "ESC_X_POB"]].sum(min_count=1)
pob["escolaridad"] = pob.pop("ESC_X_POB") / pob["P_15YMAS"]

rest = pd.read_parquet(PROC / "restaurantes_zmq.parquet")
rest = a_hex(rest[rest["edicion"] == BASE])
rest_hex = rest.pivot_table(index="h3", columns="tipo", values="id", aggfunc="count", fill_value=0)
rest_hex.columns = ["rest_" + c for c in rest_hex.columns]
rest_hex["restaurantes"] = rest_hex.sum(axis=1)
est_hex = a_hex(pd.read_parquet(PROC / f"denue_zmq_{BASE}.parquet", columns=["latitud", "longitud"])).groupby("h3").size()

hexes = pob.join([est_hex.rename("establecimientos"), rest_hex], how="outer")
conteos = [c for c in hexes.columns if c != "escolaridad"]
hexes[conteos] = hexes[conteos].fillna(0)
hexes = hexes[(hexes["POBTOT"] > 0) | (hexes["establecimientos"] > 0)]
print(f"Hexágonos con población o establecimientos: {len(hexes):,}")

## 4. Tipo de zona

Cada hexágono se compara con la mediana de población y la mediana de restaurantes (entre los hexágonos que tienen al menos uno).

| Población | Restaurantes | Tipo de zona |
|---|---|---|
| ≥ mediana | < mediana | Residencial con poca oferta |
| < mediana | ≥ mediana | Polo de restaurantes |
| ≥ mediana | ≥ mediana | Mixta |
| < mediana | < mediana | Baja densidad |

In [ ]:
p50 = hexes.loc[hexes["POBTOT"] > 0, "POBTOT"].median()
r50 = hexes.loc[hexes["restaurantes"] > 0, "restaurantes"].median()
alta_pob, alta_rest = hexes["POBTOT"] >= p50, hexes["restaurantes"] >= r50
TIPOS_ZONA = ["Residencial con poca oferta", "Polo de restaurantes", "Mixta", "Baja densidad"]
hexes["tipo_zona"] = np.select([alta_pob & ~alta_rest, ~alta_pob & alta_rest, alta_pob & alta_rest],
                               TIPOS_ZONA[:3], default=TIPOS_ZONA[3])
resumen = hexes.groupby("tipo_zona").agg(hexagonos=("tipo_zona", "size"), poblacion=("POBTOT", "sum"),
                                         restaurantes=("restaurantes", "sum")).reindex(TIPOS_ZONA)
resumen["% población"] = (resumen["poblacion"] / resumen["poblacion"].sum() * 100).round(1)
resumen["% restaurantes"] = (resumen["restaurantes"] / resumen["restaurantes"].sum() * 100).round(1)
resumen["rest. por mil hab."] = (resumen["restaurantes"] / resumen["poblacion"] * 1000).round(1)
print(f"Medianas → población: {p50:,.0f} hab · restaurantes: {r50:,.0f}")
resumen

## 5. Mapa · Tipos de zona

In [ ]:
COLORES = {"Residencial con poca oferta": "#2E6F95", "Polo de restaurantes": "#E8822F",
           "Mixta": "#14213D", "Baja densidad": "#D9D3C7"}
fig, ax, polys = mapas.base_mapa(hexes.index)
mapas.capa_hex(ax, polys, facecolors=hexes["tipo_zona"].map(COLORES).values)
mapas.referencias(ax)
ax.legend(handles=[Patch(color=c, label=f"{t} ({resumen.loc[t, 'hexagonos']:,.0f} hex.)") for t, c in COLORES.items()],
          loc="lower left", frameon=True, fontsize=11, title="Tipo de zona")
ax.set_title("Dónde vive la gente vs. dónde están los restaurantes · 2020", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "02_tipos_de_zona.png", dpi=200, bbox_inches="tight"); plt.show()

## 6. Guardar

In [ ]:
salida = PROC / f"hex_res{RES}_base_2020.parquet"
hexes.rename_axis("h3").reset_index().to_parquet(salida, index=False)
print(f"Guardado: {salida.name} · {hexes.shape[0]:,} hexágonos × {hexes.shape[1]} columnas")